In [319]:
YEAR = 2024

AIRPORT = "JFK"
# AIRPORT = "ORD"
# AIRPORT = "ATL"

PROCESSED_DATA_FILE = f"../data/bts/processed/{AIRPORT}_{YEAR}.csv"
CLEAN_DATA_FILE = f"../data/bts/cleaned/{AIRPORT}_{YEAR}.csv"

In [320]:
import pandas as pd

In [321]:
df = pd.read_csv(PROCESSED_DATA_FILE)

In [322]:
df["FlightDate"] = pd.to_datetime(df["FlightDate"], errors="coerce")

***
# Validate the cleaned BTS data

Run read-only checks before saving the file. These cells create temporary validation objects but do not change, sort, filter, or add columns to `df`.
***

***
## Check the output columns

Confirm that every expected flight, route, schedule, delay, and distance field is present. Also report any extra columns for review.
***

In [323]:
expected_columns = [
    "Year", "Quarter", "Month", "DayofMonth", "DayOfWeek",
    "FlightDate", "Reporting_Airline", "Tail_Number",
    "Flight_Number_Reporting_Airline", "Origin", "OriginState",
    "Dest", "DestState", "CRSDepTime", "DepTime", "DepDelay",
    "DepDelayMinutes", "DepDel15", "DepartureDelayGroups",
    "TaxiOut", "WheelsOff", "CRSArrTime", "ArrDel15",
    "ArrivalDelayGroups", "CRSElapsedTime", "Distance", "DistanceGroup"
]

schema_validation = pd.Series({
    "row count": len(df),
    "column count": len(df.columns),
    "missing expected columns": sorted(set(expected_columns) - set(df.columns)),
    "unexpected columns": sorted(set(df.columns) - set(expected_columns))
}, name="schema validation")

schema_validation

row count                   241356
column count                    27
missing expected columns        []
unexpected columns              []
Name: schema validation, dtype: object

***
## Check dates and calendar fields

Check the `FlightDate` type, missing values, selected year, date range, and row order. Compare the stored year, quarter, month, and day fields with the parsed flight date.
***

In [324]:
date_validation = pd.Series({
    "FlightDate dtype": str(df["FlightDate"].dtype),
    "missing flight dates": int(df["FlightDate"].isna().sum()),
    "dates outside selected year": int(
        (df["FlightDate"].notna() & (df["FlightDate"].dt.year != YEAR)).sum()
    ),
    "first flight date": df["FlightDate"].min(),
    "last flight date": df["FlightDate"].max(),
    "rows already in date order": bool(df["FlightDate"].is_monotonic_increasing)
}, name="flight date validation")

date_validation

FlightDate dtype                    datetime64[ns]
missing flight dates                             0
dates outside selected year                      0
first flight date              2024-01-01 00:00:00
last flight date               2024-12-31 00:00:00
rows already in date order                   False
Name: flight date validation, dtype: object

In [325]:
expected_quarter = ((df["FlightDate"].dt.month - 1) // 3) + 1

calendar_validation = pd.Series({
    "Year mismatches": int((df["Year"] != df["FlightDate"].dt.year).sum()),
    "Quarter mismatches": int((df["Quarter"] != expected_quarter).sum()),
    "Month mismatches": int((df["Month"] != df["FlightDate"].dt.month).sum()),
    "DayofMonth mismatches": int(
        (df["DayofMonth"] != df["FlightDate"].dt.day).sum()
    ),
    "DayOfWeek outside 1-7": int((~df["DayOfWeek"].between(1, 7)).sum())
}, name="calendar validation")

calendar_validation

Year mismatches          0
Quarter mismatches       0
Month mismatches         0
DayofMonth mismatches    0
DayOfWeek outside 1-7    0
Name: calendar validation, dtype: int64

***
## Check duplicate flights

Count fully repeated rows and repeated flight keys. The flight key combines the date, airline, flight number, route, and scheduled departure time. Duplicate keys should be reviewed before modeling.
***

In [326]:
flight_key_columns = [
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime"
]

duplicate_validation = pd.Series({
    "fully duplicated rows": int(df.duplicated().sum()),
    "rows with duplicated flight keys": int(
        df.duplicated(subset=flight_key_columns, keep=False).sum()
    )
}, name="duplicate validation")

duplicate_validation

fully duplicated rows               0
rows with duplicated flight keys    0
Name: duplicate validation, dtype: int64

***
## Check missing and numeric values

Count missing values in every column. Then confirm that expected numeric fields can be read as numbers and contain no positive or negative infinity. These checks do not fill or replace values.
***

In [327]:
missing_validation = pd.DataFrame({
    "missing count": df.isna().sum(),
    "missing percent": df.isna().mean().mul(100)
})

missing_validation

,missing count,missing percent
Year,0,0.0
Quarter,0,0.0
Month,0,0.0
DayofMonth,0,0.0
DayOfWeek,0,0.0
FlightDate,0,0.0
Reporting_Airline,0,0.0
Tail_Number,0,0.0
Flight_Number_Reporting_Airline,0,0.0
Origin,0,0.0


In [328]:
numeric_columns = [
    "Year", "Quarter", "Month", "DayofMonth", "DayOfWeek",
    "Flight_Number_Reporting_Airline", "CRSDepTime", "DepTime",
    "DepDelay", "DepDelayMinutes", "DepDel15",
    "DepartureDelayGroups", "TaxiOut", "WheelsOff", "CRSArrTime",
    "ArrDel15", "ArrivalDelayGroups", "CRSElapsedTime",
    "Distance", "DistanceGroup"
]
numeric_values = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

numeric_validation = pd.DataFrame({
    "source dtype": df[numeric_columns].dtypes.astype(str),
    "failed numeric conversion": [
        int((df[column].notna() & numeric_values[column].isna()).sum())
        for column in numeric_columns
    ],
    "infinite values": [
        int(numeric_values[column].isin([float("inf"), float("-inf")]).sum())
        for column in numeric_columns
    ]
}, index=numeric_columns)

numeric_validation

,source dtype,failed numeric conversion,infinite values
Year,int64,0,0
Quarter,int64,0,0
Month,int64,0,0
DayofMonth,int64,0,0
DayOfWeek,int64,0,0
Flight_Number_Reporting_Airline,int64,0,0
CRSDepTime,int64,0,0
DepTime,float64,0,0
DepDelay,float64,0,0
DepDelayMinutes,float64,0,0


***
## Check HHMM clock fields

BTS stores scheduled and actual clock times as HHMM numbers. Report missing, non-whole, and invalid values. `2400` is allowed as midnight; other values must use hours 00–23 and minutes 00–59.
***

In [329]:
clock_columns = ["CRSDepTime", "DepTime", "WheelsOff", "CRSArrTime"]
clock_validation_rows = []

for column in clock_columns:
    values = numeric_values[column]
    nonmissing = values.dropna()
    invalid = (
        (nonmissing < 0)
        | ((nonmissing != 2400) & (
            (nonmissing > 2359) | ((nonmissing % 100) >= 60)
        ))
    )

    clock_validation_rows.append({
        "column": column,
        "missing values": int(values.isna().sum()),
        "non-whole values": int(((nonmissing % 1) != 0).sum()),
        "invalid HHMM values": int(invalid.sum())
    })

clock_validation = pd.DataFrame(clock_validation_rows).set_index("column")
clock_validation

,missing values,non-whole values,invalid HHMM values
column,,,
CRSDepTime,0,0,0
DepTime,0,0,0
WheelsOff,0,0,0
CRSArrTime,0,0,0


***
## Check ranges and category codes

Look for impossible flight durations, distances, taxi times, or delay-minute values. Confirm that airport codes use three uppercase letters, airline codes do not need trimming or uppercasing, and each row includes the selected airport.
***

In [330]:
range_validation = pd.Series({
    "negative departure delay minutes": int((numeric_values["DepDelayMinutes"] < 0).sum()),
    "negative taxi-out times": int((numeric_values["TaxiOut"] < 0).sum()),
    "nonpositive scheduled elapsed times": int((numeric_values["CRSElapsedTime"] <= 0).sum()),
    "nonpositive distances": int((numeric_values["Distance"] <= 0).sum()),
    "invalid DepDel15 values": int((~numeric_values["DepDel15"].isin([0, 1])).sum()),
    "invalid ArrDel15 values": int((~numeric_values["ArrDel15"].isin([0, 1])).sum())
}, name="range validation")

range_validation

negative departure delay minutes       0
negative taxi-out times                0
nonpositive scheduled elapsed times    0
nonpositive distances                  0
invalid DepDel15 values                0
invalid ArrDel15 values                0
Name: range validation, dtype: int64

In [331]:
origin_codes = df["Origin"].astype("string")
destination_codes = df["Dest"].astype("string")
airline_codes = df["Reporting_Airline"].astype("string")

code_validation = pd.Series({
    "invalid origin airport codes": int(
        (~origin_codes.str.fullmatch(r"[A-Z]{3}", na=False)).sum()
    ),
    "invalid destination airport codes": int(
        (~destination_codes.str.fullmatch(r"[A-Z]{3}", na=False)).sum()
    ),
    "airline codes needing trim or uppercase": int(
        (airline_codes.notna() & airline_codes.ne(airline_codes.str.strip().str.upper())).sum()
    ),
    f"rows not involving {AIRPORT}": int(
        ((origin_codes != AIRPORT) & (destination_codes != AIRPORT)).sum()
    )
}, name="category-code validation")

code_validation

invalid origin airport codes               0
invalid destination airport codes          0
airline codes needing trim or uppercase    0
rows not involving JFK                     0
Name: category-code validation, dtype: int64

***
## Check delay labels

Compare alternate versions of the same delay result. `DepDel15` should match a departure delay of at least 15 minutes, and `ArrDel15` should match a positive arrival-delay group. Any disagreement should be reviewed before choosing the model target.
***

In [332]:
departure_label_mask = (
    numeric_values["DepDel15"].notna()
    & numeric_values["DepDelayMinutes"].notna()
)
departure_group_mask = (
    numeric_values["DepDel15"].notna()
    & numeric_values["DepartureDelayGroups"].notna()
)
arrival_label_mask = (
    numeric_values["ArrDel15"].notna()
    & numeric_values["ArrivalDelayGroups"].notna()
)
delay_minutes_mask = (
    numeric_values["DepDelay"].notna()
    & numeric_values["DepDelayMinutes"].notna()
)

target_validation = pd.Series({
    "DepDel15 vs DepDelayMinutes mismatches": int((
        departure_label_mask
        & (numeric_values["DepDel15"] != (numeric_values["DepDelayMinutes"] >= 15).astype(int))
    ).sum()),
    "DepDel15 vs DepartureDelayGroups mismatches": int((
        departure_group_mask
        & (numeric_values["DepDel15"] != (numeric_values["DepartureDelayGroups"] > 0).astype(int))
    ).sum()),
    "ArrDel15 vs ArrivalDelayGroups mismatches": int((
        arrival_label_mask
        & (numeric_values["ArrDel15"] != (numeric_values["ArrivalDelayGroups"] > 0).astype(int))
    ).sum()),
    "DepDelayMinutes vs nonnegative DepDelay mismatches": int((
        delay_minutes_mask
        & ((numeric_values["DepDelayMinutes"] - numeric_values["DepDelay"].clip(lower=0)).abs() > 1e-9)
    ).sum())
}, name="delay target validation")

target_validation

DepDel15 vs DepDelayMinutes mismatches                0
DepDel15 vs DepartureDelayGroups mismatches           0
ArrDel15 vs ArrivalDelayGroups mismatches             0
DepDelayMinutes vs nonnegative DepDelay mismatches    0
Name: delay target validation, dtype: int64

***
## Review before saving

Review missing values, duplicate keys, invalid clocks, range problems, and label mismatches before feature engineering. Remember that actual departure and arrival fields must not be used as predictors when they would be unknown at prediction time.
***

***
## Sort flights chronologically

Sort by flight date and scheduled departure time before saving. Airline and flight number provide a consistent order when several flights share the same scheduled time.
***

In [333]:
df = df.sort_values(
    [
        "FlightDate",
        "CRSDepTime",
        "Reporting_Airline",
        "Flight_Number_Reporting_Airline"
    ],
    kind="stable"
).reset_index(drop=True)

***
## Create the scheduled-departure DATE field

Combine `FlightDate` with the scheduled departure time in `CRSDepTime`. The resulting `DATE` contains the scheduled departure date, hour, and minute. A `CRSDepTime` value of `2400` becomes midnight at the start of the following day.
***

In [334]:
crs_departure_time = df["CRSDepTime"].astype(int)
scheduled_hour = crs_departure_time // 100
scheduled_minute = crs_departure_time % 100
next_day = scheduled_hour == 24
scheduled_hour = scheduled_hour.mask(next_day, 0)

df["DATE"] = (
    df["FlightDate"]
    + pd.to_timedelta(next_day.astype(int), unit="D")
    + pd.to_timedelta(scheduled_hour, unit="h")
    + pd.to_timedelta(scheduled_minute, unit="m")
)

***
## Save the cleaned and validated BTS data

Create the clean-data directory when needed and save the dataframe without the pandas index.
***

In [335]:
from pathlib import Path

Path(CLEAN_DATA_FILE).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_DATA_FILE, index=False)

In [336]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 241356 entries, 0 to 241355
Data columns (total 28 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   Year                             241356 non-null  int64         
 1   Quarter                          241356 non-null  int64         
 2   Month                            241356 non-null  int64         
 3   DayofMonth                       241356 non-null  int64         
 4   DayOfWeek                        241356 non-null  int64         
 5   FlightDate                       241356 non-null  datetime64[ns]
 6   Reporting_Airline                241356 non-null  object        
 7   Tail_Number                      241356 non-null  object        
 8   Flight_Number_Reporting_Airline  241356 non-null  int64         
 9   Origin                           241356 non-null  object        
 10  OriginState                      241356 non-

In [337]:
df.head()

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,...,DepartureDelayGroups,TaxiOut,WheelsOff,CRSArrTime,ArrDel15,ArrivalDelayGroups,CRSElapsedTime,Distance,DistanceGroup,DATE
0,2024,1,1,1,1,2024-01-01,B6,N355JB,717,BOS,...,-1.0,12.0,520.0,630,0.0,-2.0,80.0,187.0,1,2024-01-01 05:10:00
1,2024,1,1,1,1,2024-01-01,B6,N905JB,304,SJU,...,-1.0,14.0,526.0,832,0.0,-2.0,252.0,1598.0,7,2024-01-01 05:20:00
2,2024,1,1,1,1,2024-01-01,AA,N8009T,245,RDU,...,-1.0,14.0,531.0,700,0.0,-2.0,99.0,427.0,2,2024-01-01 05:21:00
3,2024,1,1,1,1,2024-01-01,B6,N807JB,838,BQN,...,0.0,14.0,547.0,830,0.0,-2.0,244.0,1576.0,7,2024-01-01 05:26:00
4,2024,1,1,1,1,2024-01-01,B6,N805JB,833,JFK,...,-1.0,13.0,530.0,836,0.0,-2.0,186.0,1028.0,5,2024-01-01 05:30:00
